In [0]:
%run ../read_params

In [0]:
%run ../utils

In [0]:
extended_national_pokedex = spark.sql(f"""
    SELECT 
        np.pokedex_number
        ,np.pokemon_name
        ,s.generation
        ,s.is_legendary
        ,s.is_mythical
        ,s.evolves_from
    FROM 
        {BRONZE_DATABASE_PREFIX}.national_pokedex np
    LEFT JOIN 
        {BRONZE_DATABASE_PREFIX}.species s
    ON 
        np.pokedex_number = s.pokedex_number
""")

extended_national_pokedex.write.format('delta').mode("overwrite").option('overwriteSchema', 'true').saveAsTable(f"{SILVER_DATABASE_PREFIX}.national_pokedex")

In [0]:
evo_chains = spark.sql(f"""
    WITH cte AS (
        SELECT 
            ec.evo_chain_id

            ,np_basic.pokedex_number AS basic_stage_pokedex_number
            ,v_basic.pokeapi_id AS basic_stage_pokeapi_id
            ,ec.basic_stage_pokemon_name
            
            ,np_stage_1.pokedex_number AS stage_1_pokedex_number
            ,v_stage_1.pokeapi_id AS stage_1_pokeapi_id
            ,ec.stage_1_pokemon_name
            
            ,np_stage_2.pokedex_number AS stage_2_pokedex_number
            ,v_stage_2.pokeapi_id AS stage_2_pokeapi_id
            ,ec.stage_2_pokemon_name
        FROM 
            bronze.evolution_chain ec

        LEFT JOIN bronze.national_pokedex np_basic
        ON ec.basic_stage_species_url = np_basic.species_url

        LEFT JOIN bronze.national_pokedex np_stage_1
        ON ec.stage_1_species_url = np_stage_1.species_url

        LEFT JOIN bronze.national_pokedex np_stage_2
        ON ec.stage_2_species_url = np_stage_2.species_url

        LEFT JOIN bronze.varieties v_basic
        ON ec.basic_stage_pokemon_name = v_basic.pokemon_name

        LEFT JOIN bronze.varieties v_stage_1
        ON ec.stage_1_pokemon_name = v_stage_1.pokemon_name

        LEFT JOIN bronze.varieties v_stage_2
        ON ec.stage_2_pokemon_name = v_stage_2.pokemon_name
    )
    SELECT 
        basic_stage_pokedex_number AS pokedex_number
        ,COALESCE(basic_stage_pokeapi_id,basic_stage_pokedex_number) AS pokeapi_id
        ,basic_stage_pokemon_name AS pokemon_name
        ,evo_chain_id
        ,NULL AS evolves_from_pokedex_number
        ,NULL AS evolves_from_pokemon_name
        ,stage_1_pokedex_number AS evolves_to_pokedex_number
        ,stage_1_pokemon_name AS evolves_to_pokemon_name
    FROM 
        cte
    WHERE 
        basic_stage_pokedex_number IS NOT NULL

    UNION

    SELECT 
        stage_1_pokedex_number AS pokedex_number
        ,COALESCE(stage_1_pokeapi_id,stage_1_pokedex_number) AS pokeapi_id
        ,stage_1_pokemon_name AS pokemon_name
        ,evo_chain_id
        ,basic_stage_pokedex_number AS evolves_from_pokedex_number
        ,basic_stage_pokemon_name AS evolves_from_pokemon_name
        ,stage_2_pokedex_number AS evolves_to_pokedex_number
        ,stage_2_pokemon_name AS evolves_to_pokemon_name
    FROM 
        cte
    WHERE 
        stage_1_pokedex_number IS NOT NULL

    UNION

    SELECT 
        stage_2_pokedex_number AS pokedex_number
        ,COALESCE(stage_2_pokeapi_id,stage_2_pokedex_number) AS pokeapi_id
        ,stage_2_pokemon_name AS pokemon_name
        ,evo_chain_id
        ,stage_1_pokedex_number AS evolves_from_pokedex_number
        ,stage_1_pokemon_name AS evolves_from_pokemon_name
        ,NULL AS evolves_to_pokedex_number
        ,NULL AS evolves_to_pokemon_name
    FROM 
        cte
    WHERE 
        stage_2_pokedex_number IS NOT NULL
""")

evo_chains.write.format('delta').mode("overwrite").option('overwriteSchema', 'true').saveAsTable(f"{SILVER_DATABASE_PREFIX}.evolution_chains")